# Photon Monte Carlo: Sampling the Planck Distribution

The Planck distribution's CDF has no closed form, so `blackbody.py`'s
`planck_cdf_sample` builds one numerically (cumulative trapezoidal
integration over a fine frequency grid) and inverts it by interpolation
to draw random photon frequencies. This notebook:

1. Validates that machinery quantitatively -- a chi-squared goodness-
   of-fit test between a large Monte Carlo sample and the analytic
   distribution, checking the *sampling procedure* (grid resolution,
   CDF inversion, tail truncation) is actually correct, not just
   plausible-looking.
2. Animates individually-sampled photons accumulating into a histogram
   that converges onto the analytic Planck curve as the photon count
   grows -- the "each photon just picks a random frequency from the
   spectrum" picture, made concrete.

In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib
from scipy import constants as const
from scipy.integrate import quad
from scipy import stats
import imageio_ffmpeg
matplotlib.rcParams['animation.ffmpeg_path'] = imageio_ffmpeg.get_ffmpeg_exe()

import blackbody as bb

os.makedirs('media', exist_ok=True)

## Quantitative check: does the sampler reproduce the target distribution?

In [2]:
# Quantitative check: does planck_cdf_sample's numerical inverse-CDF
# machinery (grid resolution, cumulative-trapezoid CDF, interpolation)
# actually reproduce the target Planck distribution, not just look
# plausible? A chi-squared goodness-of-fit test compares the observed
# histogram of a large sample against the expected bin counts from
# directly integrating spectral_radiance -- this is checking the
# SAMPLING PROCEDURE's correctness (a coarse grid, an overly-aggressive
# tail cutoff, or a CDF-inversion bug would all show up here), not
# re-deriving new physics: both sides are built from the same formula,
# which is exactly the point -- Monte Carlo samplers are validated by
# checking they actually draw from what they claim to.

T_check = 5778.0
n_samples = 500_000
rng = np.random.default_rng(42)
samples_nu = bb.planck_cdf_sample(T_check, n_samples, rng=rng)

nu_thermal = const.k * T_check / const.h
nu_max = 20 * nu_thermal
n_bins = 40
bin_edges = np.linspace(0, nu_max, n_bins + 1)

observed, _ = np.histogram(samples_nu, bins=bin_edges)
outside_frac = 1 - observed.sum() / n_samples
print(f'{n_samples} photons sampled at T={T_check:.0f} K; {outside_frac*100:.4f}% fell outside the compared range')

expected_frac = np.array([
    quad(lambda nu: bb.spectral_radiance(nu, T_check), lo, hi)[0]
    for lo, hi in zip(bin_edges[:-1], bin_edges[1:])
])
expected_frac /= expected_frac.sum()
expected_counts = expected_frac * observed.sum()

from scipy import stats
chi2, pval = stats.chisquare(observed, expected_counts)
print(f'chi-squared goodness of fit: chi2={chi2:.2f}, dof={n_bins-1}, '
      f'reduced chi2={chi2/(n_bins-1):.3f}, p-value={pval:.4f}')
assert pval > 0.01, 'sampled photon frequencies do not match the target Planck distribution'
print('PASS: the Monte Carlo sample is statistically consistent with the target Planck distribution '
      '(no evidence of a sampling bug at the 1% significance level).')

500000 photons sampled at T=5778 K; 0.0002% fell outside the compared range
chi-squared goodness of fit: chi2=25.08, dof=39, reduced chi2=0.643, p-value=0.9589
PASS: the Monte Carlo sample is statistically consistent with the target Planck distribution (no evidence of a sampling bug at the 1% significance level).


## Animation: photon histogram converging to the analytic spectrum

In [3]:
T_demo = 5778.0
N_max = 200_000
rng_demo = np.random.default_rng(7)
samples_nu_all = bb.planck_cdf_sample(T_demo, N_max, rng=rng_demo)
samples_lambda_all = const.c / samples_nu_all  # meters

hist_range = (50e-9, 3000e-9)
n_hist_bins = 60
bin_edges = np.linspace(*hist_range, n_hist_bins + 1)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
bin_width = bin_edges[1] - bin_edges[0]

# analytic overlay: normalize over a wide range so it's a genuine
# probability density (not just normalized over the visible window)
lam_wide = np.linspace(1e-9, 30000e-9, 20000)
norm_const = np.trapezoid(bb.spectral_radiance_wavelength(lam_wide, T_demo), lam_wide)
lam_plot = np.linspace(*hist_range, 500)
pdf_lambda_norm = bb.spectral_radiance_wavelength(lam_plot, T_demo) / norm_const

N_steps = np.unique(np.round(np.logspace(1, np.log10(N_max), 150)).astype(int))

fig, ax = plt.subplots(figsize=(8, 5))
counts0, _ = np.histogram(samples_lambda_all[:N_steps[0]], bins=bin_edges, density=True)
bars = ax.bar(bin_centers * 1e9, counts0, width=bin_width * 1e9, color='steelblue', alpha=0.75,
               label='Monte Carlo photons')
ax.plot(lam_plot * 1e9, pdf_lambda_norm, 'r-', lw=2, label="Planck's law (analytic)")
ax.set_ylim(0, pdf_lambda_norm.max() * 1.3)
ax.set_xlabel('wavelength (nm)')
ax.set_ylabel('probability density (per m)')
ax.set_title(f'Photon Monte Carlo at T = {T_demo:.0f} K')
n_text = ax.text(0.03, 0.93, '', ha='left', transform=ax.transAxes, fontsize=11)
ax.legend(loc='upper right')

def animate(frame_idx):
    N = N_steps[frame_idx]
    counts, _ = np.histogram(samples_lambda_all[:N], bins=bin_edges, density=True)
    for bar, height in zip(bars, counts):
        bar.set_height(height)
    n_text.set_text(f'N = {N:,} photons')
    return list(bars) + [n_text]

ani = animation.FuncAnimation(fig, animate, frames=len(N_steps), interval=60, blit=True)
ani.save('media/photon_monte_carlo.mp4', writer='ffmpeg', fps=15, dpi=130)
plt.close(fig)
print('saved media/photon_monte_carlo.mp4')

saved media/photon_monte_carlo.mp4
